# S6E9 — Competition-Only Hybrid LightGBM | CV 0.94610 / LB 0.94636

A single-model LightGBM solution for **Playground Series S6E9 — Predicting Electric Vehicle Purchases**.

Reference result from my run:

| Metric | Score |
|---|---:|
| 5-fold OOF ROC AUC | **0.946099** |
| Public LB | **0.94636** |

The leaderboard score is shown only as a reference result. It was not used to choose
features, parameters, folds, or post-processing.

---

## Credits and source lineage

This notebook is **inspired by and reimplemented from** the following public Kaggle work:

**Original notebook by [megayak](https://www.kaggle.com/megayak):**  
https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463#Ten-folds,-triple-target-encoding-refit-per-fold

The original notebook is the source that motivated the hybrid feature representation
and LightGBM recipe used here. I do **not** claim those upstream ideas as my own.

For clarity, the original notebook also gives upstream credit for part of its recipe:

- the low-order income / commute digits, modulo, quantisation ladder and exact-frequency
  idea are from **megayak's notebook above**
- the flags, triple target-encoding recipe and LightGBM recipe are credited there to
  **[najiama](https://www.kaggle.com/najiama)**  
  Source referenced by megayak:  
  https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637
- the **Smooth Keys** idea is credited there as building on
  **[maiernator](https://www.kaggle.com/maiernator)**

This notebook does not reproduce the original notebook's prose or cell structure.
The implementation below was rewritten around a competition-only, fold-local
validation pipeline.

If this notebook is useful, please also visit and upvote the original work.

---

## What is different in this version?

The original notebook reports approximately:

```text
10-fold CV : 0.94626
Public LB  : 0.94633
```

This version uses a different validation/data-flow design, so the CV values are **not
directly comparable**:

- competition data only
- no external/original-dataset target statistics
- no train+test frequency fitting
- no train+test categorical-vocabulary fitting
- frequency mappings are fitted on the current outer-training fold only
- Target Encoding is internally cross-fitted inside each outer-training fold
- outer validation is not used for LightGBM early stopping
- the number of trees is fixed at `1460`
- predictions remain ordinary `P(Yes)` probabilities
- no fold-rank post-processing

The aim is simply to see how much of the strong hybrid representation remains when
learned preprocessing is restricted to the training side of each fold.

## 1. Imports and reproducibility

In [ ]:
import gc
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

import lightgbm as lgb
import sklearn

from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder


SEED = 42


def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_everything()

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("LightGBM     :", lgb.__version__)

## 2. Configuration

In [ ]:
COMPETITION_SLUG = "playground-series-s6e9"

TARGET_COL = "Will_Buy_EV"
ID_COL = "id"

NEGATIVE_CLASS = "No"
POSITIVE_CLASS = "Yes"

TARGET_MAP = {
    NEGATIVE_CLASS: 0,
    POSITIVE_CLASS: 1,
}

N_SPLITS = 5
CV_SEED = 42

TE_N_SPLITS = 5
TE_SEED = 42

MODEL_SEED = 42
FIXED_N_ESTIMATORS = 1460

EPS = 1e-7

CATEGORICAL_FEATURES = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

NUMERICAL_FEATURES = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

EXPECTED_FEATURE_SET = set(
    CATEGORICAL_FEATURES
    + NUMERICAL_FEATURES
)

print("Target      :", TARGET_COL)
print("Positive    :", POSITIVE_CLASS)
print("Outer CV    :", N_SPLITS, "folds")
print("TE inner CV :", TE_N_SPLITS, "folds")
print("Trees       :", FIXED_N_ESTIMATORS)

## 3. Load competition data

In [ ]:
def valid_competition_dir(path: Path) -> bool:
    return all(
        (path / filename).exists()
        for filename in (
            "train.csv",
            "test.csv",
            "sample_submission.csv",
        )
    )


def resolve_competition_dir() -> Path:
    preferred = [
        Path(
            f"/kaggle/input/competitions/{COMPETITION_SLUG}"
        ),
        Path(
            f"/kaggle/input/{COMPETITION_SLUG}"
        ),
    ]

    for path in preferred:
        if valid_competition_dir(path):
            return path

    matches = []
    root = Path("/kaggle/input")

    if root.exists():
        for train_path in root.rglob("train.csv"):
            candidate = train_path.parent

            if not valid_competition_dir(candidate):
                continue

            try:
                train_cols = pd.read_csv(
                    candidate / "train.csv",
                    nrows=0,
                ).columns.tolist()

                test_cols = pd.read_csv(
                    candidate / "test.csv",
                    nrows=0,
                ).columns.tolist()

                submission_cols = pd.read_csv(
                    candidate / "sample_submission.csv",
                    nrows=0,
                ).columns.tolist()

            except Exception:
                continue

            if (
                TARGET_COL in train_cols
                and TARGET_COL not in test_cols
                and ID_COL in train_cols
                and ID_COL in test_cols
                and submission_cols == [
                    ID_COL,
                    TARGET_COL,
                ]
            ):
                matches.append(candidate)

    if len(matches) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate competition data: "
            f"{matches}"
        )

    return matches[0]


DATA_DIR = resolve_competition_dir()

train = pd.read_csv(
    DATA_DIR / "train.csv"
)

test = pd.read_csv(
    DATA_DIR / "test.csv"
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Data directory:", DATA_DIR)
print("Train:", train.shape)
print("Test :", test.shape)

## 4. Schema checks

In [ ]:
assert TARGET_COL in train.columns
assert TARGET_COL not in test.columns

assert train[ID_COL].is_unique
assert test[ID_COL].is_unique

assert not train[TARGET_COL].isna().any()

raw_classes = set(
    train[
        TARGET_COL
    ]
    .astype(str)
    .unique()
    .tolist()
)

assert raw_classes == {
    NEGATIVE_CLASS,
    POSITIVE_CLASS,
}

feature_cols = [
    col
    for col in train.columns
    if col not in [
        ID_COL,
        TARGET_COL,
    ]
]

test_feature_cols = [
    col
    for col in test.columns
    if col != ID_COL
]

assert feature_cols == test_feature_cols

assert set(
    feature_cols
) == EXPECTED_FEATURE_SET

assert sample_submission.columns.tolist() == [
    ID_COL,
    TARGET_COL,
]

assert sample_submission[
    ID_COL
].reset_index(
    drop=True
).equals(
    test[
        ID_COL
    ].reset_index(
        drop=True
    )
)

y = (
    train[
        TARGET_COL
    ]
    .map(
        TARGET_MAP
    )
    .astype(
        np.int8
    )
    .to_numpy()
)

X_raw = train[
    feature_cols
].copy()

X_test_raw = test[
    feature_cols
].copy()

print(
    train[
        TARGET_COL
    ].value_counts()
)

print(
    "Positive rate:",
    f"{y.mean():.6f}",
)

## 5. Row-local hybrid features

This section keeps the low-order income / commute representation motivated by
megayak's notebook.

These transformations are deterministic row-by-row transformations. They do not learn
anything from validation labels or from the test distribution.

The upstream attribution for these feature ideas is listed at the top of the notebook.

In [ ]:
def build_row_local_features(
    frame: pd.DataFrame,
):
    out = frame.copy()

    income = out[
        "Annual_Income_USD"
    ].to_numpy(
        dtype=np.int64
    )

    commute_x10 = np.round(
        out[
            "Daily_Commute_km"
        ].to_numpy(
            dtype=float
        )
        * 10
    ).astype(
        np.int64
    )

    # Low-order income / commute structure.
    # Feature idea lineage: megayak (credited above).
    out["inc_d1"] = (
        income % 10
    ).astype("int8")

    out["inc_d2"] = (
        income // 10 % 10
    ).astype("int8")

    out["inc_d3"] = (
        income // 100 % 10
    ).astype("int8")

    out["inc_mod100"] = (
        income % 100
    ).astype("int16")

    out["inc_mod1000"] = (
        income % 1000
    ).astype("int16")

    out["km_d1"] = (
        commute_x10 % 10
    ).astype("int8")

    out["km_mod100"] = (
        commute_x10 % 100
    ).astype("int8")

    for divisor in (
        50,
        100,
        250,
        500,
        1000,
        2500,
        5000,
    ):
        out[
            f"inc_q{divisor}"
        ] = (
            income // divisor
        ).astype(
            "int32"
        )

    for divisor in (
        5,
        10,
        25,
        50,
    ):
        out[
            f"km_q{divisor}"
        ] = (
            commute_x10 // divisor
        ).astype(
            "int32"
        )

    # Flag lineage: credited by megayak to najiama.
    out[
        "is_30k_spike"
    ] = (
        income == 30000
    ).astype(
        "int8"
    )

    out[
        "is_millionaire_cliff"
    ] = (
        income >= 170537
    ).astype(
        "int8"
    )

    out[
        "is_dead_zone"
    ] = (
        (
            income >= 38000
        )
        & (
            income <= 42000
        )
    ).astype(
        "int8"
    )

    out[
        "is_env_hater"
    ] = (
        out[
            "Environmental_Concern_Level"
        ]
        == 1
    ).astype(
        "int8"
    )

    # Smooth-Key family:
    # credited by megayak to najiama / maiernator.
    keys = pd.DataFrame(
        index=out.index
    )

    keys[
        "k_inc_exact"
    ] = income.astype(
        str
    )

    keys[
        "k_inc100"
    ] = (
        income // 100
    ).astype(
        str
    )

    keys[
        "k_inc1000"
    ] = (
        income // 1000
    ).astype(
        str
    )

    keys[
        "k_km_int"
    ] = (
        commute_x10 // 10
    ).astype(
        str
    )

    for col in (
        CATEGORICAL_FEATURES
        + [
            "Age",
            "Number_of_Cars_Owned",
            "Charging_Stations_Near_Home",
            "Charging_Stations_Near_Work",
            "Environmental_Concern_Level",
        ]
    ):
        keys[
            f"k_{col}"
        ] = (
            out[
                col
            ]
            .astype(
                str
            )
            .to_numpy()
        )

    return (
        out.reset_index(
            drop=True
        ),
        keys.reset_index(
            drop=True
        ),
    )


X_rowlocal, X_keys = (
    build_row_local_features(
        X_raw
    )
)

X_test_rowlocal, X_test_keys = (
    build_row_local_features(
        X_test_raw
    )
)

assert X_rowlocal.shape[
    1
] == 35

assert X_keys.shape[
    1
] == 15

print(
    "Row-local features:",
    X_rowlocal.shape[
        1
    ],
)

print(
    "Encoding keys:",
    X_keys.shape[
        1
    ],
)

## 6. Fold-local frequency + triple Target Encoding

All learned preprocessing is fitted from the current outer-training fold.

For every outer fold:

- `fq_inc` and `fq_km` counts come from outer-training rows only
- key-frequency mappings come from outer-training rows only
- categorical vocabularies come from outer-training rows only
- the outer-training Target Encoding is generated with
  `TargetEncoder.fit_transform`, using internal cross-fitting
- outer validation and test are transformed from an encoder fitted only on the
  outer-training fold

The three Target-Encoding views are:

```text
smooth = auto
smooth = 10
smooth = 100
```

This follows the triple-encoding idea credited in the source lineage above, but the
fit scope here is deliberately fold-local.

In [ ]:
def as_string(
    series: pd.Series,
) -> pd.Series:
    return (
        series
        .astype(
            str
        )
        .reset_index(
            drop=True
        )
    )


def map_with_default_zero(
    values: pd.Series,
    mapping: pd.Series,
) -> np.ndarray:
    return (
        values
        .map(
            mapping
        )
        .fillna(
            0.0
        )
        .astype(
            np.float32
        )
        .to_numpy()
    )


def prepare_fold_features(
    train_positions,
    valid_positions,
):
    train_part = (
        X_rowlocal
        .iloc[
            train_positions
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    valid_part = (
        X_rowlocal
        .iloc[
            valid_positions
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    test_part = (
        X_test_rowlocal
        .reset_index(
            drop=True
        )
        .copy()
    )

    train_keys = (
        X_keys
        .iloc[
            train_positions
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    valid_keys = (
        X_keys
        .iloc[
            valid_positions
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    test_keys = (
        X_test_keys
        .reset_index(
            drop=True
        )
        .copy()
    )

    # Exact income frequency:
    # mapping is fitted only on the current outer-training fold.
    income_train = pd.Series(
        train_part[
            "Annual_Income_USD"
        ].to_numpy(
            dtype=np.int64
        )
    )

    income_valid = pd.Series(
        valid_part[
            "Annual_Income_USD"
        ].to_numpy(
            dtype=np.int64
        )
    )

    income_test = pd.Series(
        test_part[
            "Annual_Income_USD"
        ].to_numpy(
            dtype=np.int64
        )
    )

    income_counts = (
        income_train
        .value_counts(
            dropna=False
        )
    )

    train_part[
        "fq_inc"
    ] = map_with_default_zero(
        income_train,
        income_counts,
    )

    valid_part[
        "fq_inc"
    ] = map_with_default_zero(
        income_valid,
        income_counts,
    )

    test_part[
        "fq_inc"
    ] = map_with_default_zero(
        income_test,
        income_counts,
    )

    # Exact commute frequency:
    # use the same x10 integer representation as the row-local features.
    commute_train = pd.Series(
        np.round(
            train_part[
                "Daily_Commute_km"
            ].to_numpy(
                dtype=float
            )
            * 10
        ).astype(
            np.int64
        )
    )

    commute_valid = pd.Series(
        np.round(
            valid_part[
                "Daily_Commute_km"
            ].to_numpy(
                dtype=float
            )
            * 10
        ).astype(
            np.int64
        )
    )

    commute_test = pd.Series(
        np.round(
            test_part[
                "Daily_Commute_km"
            ].to_numpy(
                dtype=float
            )
            * 10
        ).astype(
            np.int64
        )
    )

    commute_counts = (
        commute_train
        .value_counts(
            dropna=False
        )
    )

    train_part[
        "fq_km"
    ] = map_with_default_zero(
        commute_train,
        commute_counts,
    )

    valid_part[
        "fq_km"
    ] = map_with_default_zero(
        commute_valid,
        commute_counts,
    )

    test_part[
        "fq_km"
    ] = map_with_default_zero(
        commute_test,
        commute_counts,
    )

    # Normalized Smooth-Key frequencies.
    for key in train_keys.columns:
        frequency_map = (
            train_keys[
                key
            ]
            .value_counts(
                normalize=True,
                dropna=False,
            )
        )

        feature_name = (
            f"{key}_fe"
        )

        train_part[
            feature_name
        ] = map_with_default_zero(
            train_keys[
                key
            ],
            frequency_map,
        )

        valid_part[
            feature_name
        ] = map_with_default_zero(
            valid_keys[
                key
            ],
            frequency_map,
        )

        test_part[
            feature_name
        ] = map_with_default_zero(
            test_keys[
                key
            ],
            frequency_map,
        )

    # Categorical vocabulary is also taken from outer training only.
    for col in CATEGORICAL_FEATURES:
        train_strings = (
            as_string(
                train_part[
                    col
                ]
            )
        )

        valid_strings = (
            as_string(
                valid_part[
                    col
                ]
            )
        )

        test_strings = (
            as_string(
                test_part[
                    col
                ]
            )
        )

        categories = sorted(
            train_strings
            .unique()
            .tolist()
        )

        category_set = set(
            categories
        )

        unseen_valid = (
            set(
                valid_strings
                .unique()
                .tolist()
            )
            - category_set
        )

        unseen_test = (
            set(
                test_strings
                .unique()
                .tolist()
            )
            - category_set
        )

        if (
            unseen_valid
            or unseen_test
        ):
            raise RuntimeError(
                f"Unseen category in {col}: "
                f"valid={sorted(unseen_valid)}, "
                f"test={sorted(unseen_test)}"
            )

        dtype = (
            pd.CategoricalDtype(
                categories=categories,
                ordered=False,
            )
        )

        train_part[
            col
        ] = (
            train_strings
            .astype(
                dtype
            )
        )

        valid_part[
            col
        ] = (
            valid_strings
            .astype(
                dtype
            )
        )

        test_part[
            col
        ] = (
            test_strings
            .astype(
                dtype
            )
        )

    # Triple Target Encoding.
    y_train = y[
        train_positions
    ]

    for smooth, tag in (
        (
            "auto",
            "auto",
        ),
        (
            10.0,
            "10",
        ),
        (
            100.0,
            "100",
        ),
    ):
        encoder = (
            TargetEncoder(
                shuffle=True,
                cv=TE_N_SPLITS,
                smooth=smooth,
                random_state=TE_SEED,
            )
        )

        encoded_train = (
            encoder
            .fit_transform(
                train_keys,
                y_train,
            )
        )

        encoded_valid = (
            encoder
            .transform(
                valid_keys
            )
        )

        encoded_test = (
            encoder
            .transform(
                test_keys
            )
        )

        assert np.isfinite(
            encoded_train
        ).all()

        assert np.isfinite(
            encoded_valid
        ).all()

        assert np.isfinite(
            encoded_test
        ).all()

        for idx, key in enumerate(
            train_keys.columns
        ):
            feature_name = (
                f"{key}_te{tag}"
            )

            train_part[
                feature_name
            ] = (
                encoded_train[
                    :,
                    idx,
                ]
                .astype(
                    np.float32
                )
            )

            valid_part[
                feature_name
            ] = (
                encoded_valid[
                    :,
                    idx,
                ]
                .astype(
                    np.float32
                )
            )

            test_part[
                feature_name
            ] = (
                encoded_test[
                    :,
                    idx,
                ]
                .astype(
                    np.float32
                )
            )

    assert train_part.shape[
        1
    ] == 97

    assert valid_part.shape[
        1
    ] == 97

    assert test_part.shape[
        1
    ] == 97

    assert (
        train_part.columns.tolist()
        == valid_part.columns.tolist()
        == test_part.columns.tolist()
    )

    return (
        train_part,
        valid_part,
        test_part,
    )

## 7. LightGBM

In [ ]:
def make_model():
    return lgb.LGBMClassifier(
        n_estimators=FIXED_N_ESTIMATORS,
        learning_rate=0.02,
        max_depth=5,
        num_leaves=32,
        min_child_samples=10,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.3,
        reg_alpha=0.071,
        reg_lambda=2.0,
        max_bin=1024,
        feature_pre_filter=False,
        random_state=MODEL_SEED,
        n_jobs=-1,
        verbose=-1,
    )


print(
    make_model()
)

## 8. Five-fold training

The outer validation fold is used only for scoring.

It is not passed as an `eval_set`, so it does not choose the number of boosting rounds.
Every fold uses the same `1460` trees.

In [ ]:
outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_SEED,
)

splits = list(
    outer_cv.split(
        np.zeros(
            len(y),
            dtype=np.int8,
        ),
        y,
    )
)

oof = np.full(
    len(train),
    np.nan,
    dtype=np.float64,
)

oof_count = np.zeros(
    len(train),
    dtype=np.int8,
)

fold_ids = np.full(
    len(train),
    -1,
    dtype=np.int8,
)

test_pred_folds = np.zeros(
    (
        N_SPLITS,
        len(test),
    ),
    dtype=np.float64,
)

fold_rows = []
importance_rows = []

start_all = time.time()

for fold, (
    train_idx,
    valid_idx,
) in enumerate(
    splits
):
    fold_start = time.time()

    print(
        f"\n===== Fold {fold + 1}/{N_SPLITS} ====="
    )

    (
        X_train_fold,
        X_valid_fold,
        X_test_fold,
    ) = (
        prepare_fold_features(
            train_idx,
            valid_idx,
        )
    )

    seed_everything(
        MODEL_SEED
    )

    model = (
        make_model()
    )

    # Outer validation is intentionally not passed to model.fit().
    model.fit(
        X_train_fold,
        y[
            train_idx
        ],
    )

    classes = (
        model.classes_
        .tolist()
    )

    assert classes == [
        0,
        1,
    ]

    positive_index = (
        classes.index(
            1
        )
    )

    valid_pred = (
        model
        .predict_proba(
            X_valid_fold
        )[
            :,
            positive_index,
        ]
        .astype(
            np.float64
        )
    )

    test_pred = (
        model
        .predict_proba(
            X_test_fold
        )[
            :,
            positive_index,
        ]
        .astype(
            np.float64
        )
    )

    assert np.isfinite(
        valid_pred
    ).all()

    assert np.isfinite(
        test_pred
    ).all()

    assert np.all(
        (
            valid_pred >= 0.0
        )
        & (
            valid_pred <= 1.0
        )
    )

    assert np.all(
        (
            test_pred >= 0.0
        )
        & (
            test_pred <= 1.0
        )
    )

    fold_auc = float(
        roc_auc_score(
            y[
                valid_idx
            ],
            valid_pred,
        )
    )

    fold_logloss = float(
        log_loss(
            y[
                valid_idx
            ],
            np.clip(
                valid_pred,
                EPS,
                1.0 - EPS,
            ),
            labels=[
                0,
                1,
            ],
        )
    )

    assert np.all(
        oof_count[
            valid_idx
        ] == 0
    )

    oof[
        valid_idx
    ] = valid_pred

    oof_count[
        valid_idx
    ] += 1

    fold_ids[
        valid_idx
    ] = fold

    test_pred_folds[
        fold
    ] = test_pred

    fold_rows.append({
        "fold": fold + 1,
        "auc": fold_auc,
        "log_loss": fold_logloss,
        "pred_mean": float(
            valid_pred.mean()
        ),
        "pred_std": float(
            valid_pred.std()
        ),
        "seconds": float(
            time.time()
            - fold_start
        ),
    })

    gain = (
        model
        .booster_
        .feature_importance(
            importance_type="gain"
        )
    )

    feature_names = (
        model
        .booster_
        .feature_name()
    )

    importance_rows.append(
        pd.DataFrame({
            "fold": fold + 1,
            "feature": feature_names,
            "gain": gain.astype(
                np.float64
            ),
        })
    )

    print(
        f"AUC={fold_auc:.6f} | "
        f"LogLoss={fold_logloss:.6f} | "
        f"{time.time() - fold_start:.1f}s"
    )

    del (
        X_train_fold,
        X_valid_fold,
        X_test_fold,
        model,
        valid_pred,
        test_pred,
    )

    gc.collect()

assert np.all(
    oof_count == 1
)

assert np.all(
    fold_ids >= 0
)

assert np.isfinite(
    oof
).all()

test_pred = (
    test_pred_folds
    .mean(
        axis=0
    )
)

assert np.isfinite(
    test_pred
).all()

print(
    "\nTotal minutes:",
    f"{(time.time() - start_all) / 60.0:.2f}",
)

## 9. Cross-validation result

In [ ]:
fold_metrics = pd.DataFrame(
    fold_rows
)

overall_auc = float(
    roc_auc_score(
        y,
        oof,
    )
)

overall_logloss = float(
    log_loss(
        y,
        np.clip(
            oof,
            EPS,
            1.0 - EPS,
        ),
        labels=[
            0,
            1,
        ],
    )
)

display(
    fold_metrics
)

print()

print(
    f"OOF ROC AUC : {overall_auc:.9f}"
)

print(
    f"OOF LogLoss : {overall_logloss:.9f}"
)

print(
    "Fold AUC mean/std:",
    f"{fold_metrics['auc'].mean():.9f}",
    "/",
    f"{fold_metrics['auc'].std(ddof=0):.9f}",
)

## 10. What did the model use?

This is descriptive only; feature importance is not used to tune or select the model.

It is useful to check whether the multiple income-resolution encodings, frequency
features, and row-local structure remain active after using the fold-local pipeline.

In [ ]:
importance = pd.concat(
    importance_rows,
    ignore_index=True,
)

importance_summary = (
    importance
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        gain_mean=(
            "gain",
            "mean",
        ),
        gain_std=(
            "gain",
            "std",
        ),
    )
    .sort_values(
        "gain_mean",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


def feature_family(
    feature_name: str,
) -> str:
    if "_teauto" in feature_name:
        return "Target Encoding: auto"

    if "_te10" in feature_name:
        return "Target Encoding: 10"

    if "_te100" in feature_name:
        return "Target Encoding: 100"

    if feature_name.endswith(
        "_fe"
    ):
        return "Key frequency"

    if feature_name in {
        "fq_inc",
        "fq_km",
    }:
        return "Exact frequency"

    if (
        feature_name.startswith(
            "inc_"
        )
        or feature_name.startswith(
            "km_"
        )
        or feature_name.startswith(
            "is_"
        )
    ):
        return "Row-local structure"

    return "Raw"


importance_summary[
    "family"
] = (
    importance_summary[
        "feature"
    ]
    .map(
        feature_family
    )
)

family_gain = (
    importance_summary
    .groupby(
        "family",
        as_index=False,
    )[
        "gain_mean"
    ]
    .sum()
)

family_gain[
    "gain_share"
] = (
    family_gain[
        "gain_mean"
    ]
    / family_gain[
        "gain_mean"
    ].sum()
)

display(
    importance_summary.head(
        25
    )
)

display(
    family_gain.sort_values(
        "gain_share",
        ascending=False,
    )
)

## 11. Save OOF, test predictions, and submission

Three compact CSV files are written:

- `oof_predictions.csv`
  - original `id`
  - outer-fold number
  - out-of-fold `P(Yes)`
- `test_predictions.csv`
  - original `id`
  - five-fold averaged `P(Yes)`
- `submission.csv`
  - Kaggle submission format

In [ ]:
oof_predictions = pd.DataFrame({
    ID_COL: train[
        ID_COL
    ].to_numpy(),

    "fold": fold_ids.astype(
        np.int8
    ),

    TARGET_COL: oof.astype(
        np.float64
    ),
})

test_predictions = pd.DataFrame({
    ID_COL: test[
        ID_COL
    ].to_numpy(),

    TARGET_COL: test_pred.astype(
        np.float64
    ),
})

submission = (
    sample_submission
    .copy()
)

submission[
    TARGET_COL
] = (
    test_predictions[
        TARGET_COL
    ].to_numpy()
)

assert len(
    oof_predictions
) == len(
    train
)

assert len(
    test_predictions
) == len(
    test
)

assert submission.shape == (
    len(test),
    2,
)

assert submission.columns.tolist() == [
    ID_COL,
    TARGET_COL,
]

assert (
    submission[
        ID_COL
    ]
    .reset_index(
        drop=True
    )
    .equals(
        test[
            ID_COL
        ]
        .reset_index(
            drop=True
        )
    )
)

for frame in (
    oof_predictions,
    test_predictions,
    submission,
):
    assert np.isfinite(
        frame[
            TARGET_COL
        ].to_numpy()
    ).all()

    assert frame[
        TARGET_COL
    ].between(
        0.0,
        1.0,
    ).all()

OUTPUT_DIR = Path(
    "/kaggle/working"
)

OOF_FILE = (
    OUTPUT_DIR
    / "oof_predictions.csv"
)

TEST_PRED_FILE = (
    OUTPUT_DIR
    / "test_predictions.csv"
)

SUBMISSION_FILE = (
    OUTPUT_DIR
    / "submission.csv"
)

oof_predictions.to_csv(
    OOF_FILE,
    index=False,
)

test_predictions.to_csv(
    TEST_PRED_FILE,
    index=False,
)

submission.to_csv(
    SUBMISSION_FILE,
    index=False,
)

print("Saved:")
print(" ", OOF_FILE)
print(" ", TEST_PRED_FILE)
print(" ", SUBMISSION_FILE)

display(
    submission.head()
)

## Takeaways

The main observation from this competition-only version is simple:

- the hybrid income / commute representation remains strong without external
  target-statistic features
- fold-local frequency mappings and internally cross-fitted Target Encoding preserve
  a clean outer-validation path
- the multiple Smooth-Key resolutions remain useful
- a fixed boosting budget avoids using the outer validation fold to choose the
  checkpoint
- the resulting single LightGBM reached **0.946099 OOF AUC** and **0.94636 Public LB**
  in my run

The Public LB happened to be slightly higher than the original notebook's reported
`0.94633`, while this notebook's OOF CV is lower. Because the validation designs differ,
I would not interpret that as evidence that this version is universally better.

The most important credit remains with the upstream work that motivated the recipe:
**megayak**, **najiama**, and **maiernator**. Please visit the linked notebooks/profiles
at the top if you found this useful.